In [ ]:
# === auto-inserted: bev-solving src on path ===
import sys, pathlib
_root = pathlib.Path.cwd()
while _root != _root.parent and not (_root / 'src' / 'geometry.py').exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


# Train BEV v4 Small Rover Embeddings, No Cleaning, 2 GPU

Self-contained notebook for a clean ablation of the previous pipeline:
- merge `train + val` into one pool;
- no empty-GT filtering;
- no deduplication;
- test-matched grouped validation targeting about 200 samples;
- compact rover embeddings with `Other=0`, top-25 frequent train rovers getting unique IDs;
- stronger weight decay only on the embedding table;
- optional 2-GPU training with `nn.DataParallel` when the runtime exposes more than one CUDA device.

This notebook does not modify the original dataset folders. It writes only caches and checkpoints into `RUN_DIR`.


In [ ]:
%load_ext autoreload
%autoreload 2

import os, copy, time, json, random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageFile
from tqdm import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

DATA_TRAIN = Path('./autonomy_yandex_dataset_train/')
DATA_VAL   = Path('./autonomy_yandex_dataset_val/')
DATA_TEST  = Path('./autonomy_yandex_dataset_test/')

cfg = {
    'run_dir': './runs/v4_small_rover_2gpu',
    'img_hw': (384, 704),
    'batch_size': 8,
    'grad_accum': 4,
    'epochs': 20,
    'warmup_epochs': 3,
    'lr_backbone': 3e-5,
    'lr_head': 3e-4,
    'weight_decay': 1e-4,
    'embed_weight_decay': 1e-2,
    'pos_weight': 5.0,
    'num_workers': 4,
    'seed': 42,
    'ema_decay': 0.995,
    'val_target_size': 200,
    'min_rover_count': 30,
    'topk_rovers': 25,
    'rover_emb_dim': 8,
    'rover_cond_dim': 8,
    'use_dp_if_available': True,
}

RUN_DIR = Path(cfg['run_dir'])
RUN_DIR.mkdir(parents=True, exist_ok=True)

random.seed(cfg['seed'])
np.random.seed(cfg['seed'])
torch.manual_seed(cfg['seed'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_cuda = torch.cuda.device_count() if torch.cuda.is_available() else 0
print('device:', device)
print('cuda device count:', num_cuda)
if num_cuda:
    for i in range(num_cuda):
        print(i, torch.cuda.get_device_name(i))

with open(RUN_DIR / 'config.json', 'w') as f:
    json.dump({k: str(v) for k, v in cfg.items()}, f, indent=2)


In [ ]:
from src.geometry import load_info_with_root, resolve_info_path
from src.splits import build_rover_vocab_from_train, encode_rover, make_test_matched_split_target

CAMERA_NAMES = [
    '/camera/inner/frontal/middle',
    '/camera/inner/frontal/far',
    '/side/left/forward',
    '/side/right/forward',
]
INTRINSICS_NAMES = [n + '/intrinsic_params' for n in CAMERA_NAMES]
CAR2CAM_NAMES = [n + '/car_to_cam' for n in CAMERA_NAMES]
GT_NAME = 'gt_occupancy_grid'

BEV_H, BEV_W = 188, 126
BEV_RES = 0.8
X_RANGE = (0.0, BEV_H * BEV_RES)
Y_RANGE = (-BEV_W * BEV_RES / 2, BEV_W * BEV_RES / 2)
Z_LEVELS = (0.3, 1.0, 2.0, 3.0)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


In [ ]:
merged_info = pd.concat([
    load_info_with_root(DATA_TRAIN, 'train'),
    load_info_with_root(DATA_VAL, 'val'),
], ignore_index=True)

train_idx, val_idx = make_test_matched_split_target(
    merged_info,
    DATA_TEST / 'info.csv',
    target_val_size=cfg['val_target_size'],
    seed=cfg['seed'],
    cache_path=RUN_DIR / 'test_matched_val200_noclean_split.npz',
)

train_info = merged_info.iloc[train_idx].reset_index(drop=True).copy()
val_info = merged_info.iloc[val_idx].reset_index(drop=True).copy()

rover_vocab, rover_stats = build_rover_vocab_from_train(
    train_info,
    min_count=cfg['min_rover_count'],
    topk=cfg['topk_rovers'],
)
rover_stats.to_csv(RUN_DIR / 'rover_embedding_stats.csv', index=False)

print('merged total:', len(merged_info))
print('train size:', len(train_info))
print('val size:', len(val_info))
print('num rover classes including Other:', len(rover_vocab))
display(rover_stats.head(30))


In [ ]:
class BEVDatasetV4SmallRover(Dataset):
    def __init__(self, info_df: pd.DataFrame, mode: str = 'train',
                 img_hw=(384, 704), aug: bool = False,
                 rover_vocab: dict | None = None):
        self.info = info_df.reset_index(drop=True).copy()
        self.mode = mode
        self.img_hw = img_hw
        self.aug = aug and (mode == 'train')
        self.rover_vocab = rover_vocab or {'__other__': 0}
        self.normalize = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)

    def __len__(self):
        return len(self.info)

    def _resolve_path(self, row: pd.Series, key: str) -> Path:
        return resolve_info_path(Path(row['__data_root']), row[key])

    def _load_one_camera(self, row: pd.Series, cam_key: str, intr_key: str, c2c_key: str,
                         scale_aug: float = 1.0):
        img = Image.open(self._resolve_path(row, cam_key)).convert('RGB')
        intr_path = self._resolve_path(row, intr_key)
        car2cam_path = self._resolve_path(row, c2c_key)
        src_W, src_H = img.size
        H_t, W_t = self.img_hw

        s = min(W_t / src_W, H_t / src_H)
        new_W, new_H = int(round(src_W * s)), int(round(src_H * s))
        img_resized = img.resize((new_W, new_H), Image.BILINEAR)
        canvas = Image.new('RGB', (W_t, H_t), (0, 0, 0))
        pad_x = (W_t - new_W) // 2
        pad_y = (H_t - new_H) // 2
        canvas.paste(img_resized, (pad_x, pad_y))

        extra_s = 1.0
        extra_dx = 0
        extra_dy = 0
        if scale_aug > 1.0:
            sH, sW = int(round(H_t * scale_aug)), int(round(W_t * scale_aug))
            canvas = canvas.resize((sW, sH), Image.BILINEAR)
            extra_dx = random.randint(0, sW - W_t)
            extra_dy = random.randint(0, sH - H_t)
            canvas = canvas.crop((extra_dx, extra_dy, extra_dx + W_t, extra_dy + H_t))
            extra_s = scale_aug

        arr = np.array(canvas)
        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)
        img_t = torch.from_numpy(arr).permute(2, 0, 1).float() / 255.0
        img_t = self.normalize(img_t)

        intr_full = np.load(intr_path)
        K = intr_full[:, :3].copy().astype(np.float32)
        K[0, 0] *= s; K[0, 2] *= s
        K[1, 1] *= s; K[1, 2] *= s
        K[0, 2] += pad_x
        K[1, 2] += pad_y
        K[0, 0] *= extra_s; K[0, 2] *= extra_s
        K[1, 1] *= extra_s; K[1, 2] *= extra_s
        K[0, 2] -= extra_dx
        K[1, 2] -= extra_dy

        car2cam = np.load(car2cam_path).astype(np.float32)
        return img_t, K, car2cam

    def _load_sample(self, idx: int):
        row = self.info.iloc[idx]
        scale_aug = random.uniform(1.0, 1.15) if self.aug else 1.0

        imgs, Ks, M = [], [], []
        for cam, intr, c2c in zip(CAMERA_NAMES, INTRINSICS_NAMES, CAR2CAM_NAMES):
            img_t, K, c = self._load_one_camera(row, cam, intr, c2c, scale_aug=scale_aug)
            imgs.append(img_t)
            Ks.append(torch.from_numpy(K))
            M.append(torch.from_numpy(c))

        out = {
            'images': torch.stack(imgs, dim=0),
            'intrinsics': torch.stack(Ks, dim=0),
            'car2cams': torch.stack(M, dim=0),
            'rover_id': torch.tensor(encode_rover(row.get('rover', '__other__'), self.rover_vocab), dtype=torch.long),
            'info_idx': idx,
        }
        if self.mode == 'test':
            return out
        gt = np.load(self._resolve_path(row, GT_NAME)).squeeze()
        gt = np.where(gt < 0, 255, gt).astype(np.int64)
        out['gt'] = torch.from_numpy(gt).unsqueeze(0)
        return out

    def __getitem__(self, idx: int):
        max_tries = 5
        last_err = None
        for k in range(max_tries):
            try_idx = (idx + k) % len(self.info)
            try:
                return self._load_sample(try_idx)
            except (OSError, ValueError, FileNotFoundError) as e:
                last_err = e
                continue
        raise RuntimeError(f'Failed to load sample after {max_tries} tries from idx={idx}: {last_err}')


In [ ]:
from src.models.backbones import _ResNet50Backbone
from src.models.decoder import SmallUNet, _UNetBlock


class MultiCamBEVv4SmallRover(nn.Module):
    def __init__(self, num_rover_classes: int,
                 rover_emb_dim: int = 8,
                 rover_cond_dim: int = 8,
                 n_cameras: int = 4,
                 freeze_backbone: bool = False):
        super().__init__()
        self.n_cameras = n_cameras
        self.bev_h, self.bev_w = BEV_H, BEV_W
        self.x_range, self.y_range = X_RANGE, Y_RANGE
        self.z_levels = Z_LEVELS
        self.rover_cond_dim = rover_cond_dim

        self.backbone = _ResNet50Backbone(pretrained=True)
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        self.feat_proj = nn.Conv2d(128, 64, 1)
        self.register_buffer('ego_voxels', self._make_ego_voxels(), persistent=False)

        self.rover_embed = nn.Embedding(num_rover_classes, rover_emb_dim)
        nn.init.normal_(self.rover_embed.weight, std=0.02)
        self.rover_mlp = nn.Sequential(
            nn.Linear(rover_emb_dim, 16),
            nn.ReLU(inplace=True),
            nn.Linear(16, rover_cond_dim),
            nn.ReLU(inplace=True),
        )

        in_c = 64 * len(self.z_levels) + rover_cond_dim
        self.bev_decoder = SmallUNet(in_c=in_c, base_c=32, out_c=1)

    def _make_ego_voxels(self):
        xs = torch.linspace(self.x_range[0] + BEV_RES / 2, self.x_range[1] - BEV_RES / 2, self.bev_h)
        ys = torch.linspace(self.y_range[0] + BEV_RES / 2, self.y_range[1] - BEV_RES / 2, self.bev_w)
        zs = torch.tensor(self.z_levels, dtype=torch.float32)
        Z, X, Y = torch.meshgrid(zs, xs, ys, indexing='ij')
        ones = torch.ones_like(X)
        return torch.stack([X, Y, Z, ones], dim=-1)

    def forward(self, images, intrinsics, car2cams, rover_ids):
        B, N, C_, Hi, Wi = images.shape
        assert N == self.n_cameras

        feat = self.backbone(images.reshape(B * N, C_, Hi, Wi))
        feat = self.feat_proj(feat)
        Hf, Wf = feat.shape[-2:]
        feat = feat.reshape(B, N, 64, Hf, Wf)

        Z, H, W, _ = self.ego_voxels.shape
        V = Z * H * W
        voxels = self.ego_voxels.reshape(-1, 4).unsqueeze(0).unsqueeze(0).expand(B, N, -1, -1)

        p_cam = torch.einsum('bnij,bnvj->bniv', car2cams, voxels)
        p_cam_3d = p_cam[:, :, :3]
        uv_homog = torch.einsum('bnij,bnjv->bniv', intrinsics, p_cam_3d)
        z = uv_homog[:, :, 2]
        u = uv_homog[:, :, 0] / z.clamp(min=1e-3)
        v = uv_homog[:, :, 1] / z.clamp(min=1e-3)

        u_n = 2.0 * u / Wi - 1.0
        v_n = 2.0 * v / Hi - 1.0
        valid = (z > 0.1) & (u_n.abs() <= 1.0) & (v_n.abs() <= 1.0)

        grid = torch.stack([u_n, v_n], dim=-1).reshape(B * N, V, 1, 2)
        sampled = F.grid_sample(
            feat.reshape(B * N, 64, Hf, Wf),
            grid,
            mode='bilinear',
            padding_mode='zeros',
            align_corners=False,
        )
        sampled = sampled.squeeze(-1).reshape(B, N, 64, V)

        valid_f = valid.float().unsqueeze(2)
        sampled = sampled * valid_f
        agg = sampled.sum(dim=1) / valid_f.sum(dim=1).clamp(min=1.0)
        agg = agg.reshape(B, 64, Z, H, W).reshape(B, 64 * Z, H, W)

        rover_feat = self.rover_mlp(self.rover_embed(rover_ids)).view(B, self.rover_cond_dim, 1, 1)
        rover_map = rover_feat.expand(-1, -1, H, W)
        agg = torch.cat([agg, rover_map], dim=1)
        return self.bev_decoder(agg)


In [ ]:
from src.eval import evaluate_iou
from src.losses import CompoundLossV2, _lovasz_grad, lovasz_hinge_flat
from src.metrics import iou_binary_batch, streaming_threshold_sweep
from src.utils.training import unwrap_model, update_ema



In [ ]:
ds_train_warm = BEVDatasetV4SmallRover(train_info, mode='train', img_hw=cfg['img_hw'], aug=False, rover_vocab=rover_vocab)
ds_train_aug = BEVDatasetV4SmallRover(train_info, mode='train', img_hw=cfg['img_hw'], aug=True, rover_vocab=rover_vocab)
ds_val = BEVDatasetV4SmallRover(val_info, mode='val', img_hw=cfg['img_hw'], aug=False, rover_vocab=rover_vocab)

loader_warm = DataLoader(ds_train_warm, batch_size=cfg['batch_size'], shuffle=True,
                         num_workers=cfg['num_workers'], pin_memory=(device.type == 'cuda'), drop_last=True)
loader_train = DataLoader(ds_train_aug, batch_size=cfg['batch_size'], shuffle=True,
                          num_workers=cfg['num_workers'], pin_memory=(device.type == 'cuda'), drop_last=True)
loader_val = DataLoader(ds_val, batch_size=cfg['batch_size'], shuffle=False,
                        num_workers=cfg['num_workers'], pin_memory=(device.type == 'cuda'))

sample = ds_train_warm[0]
for k, v in sample.items():
    if isinstance(v, torch.Tensor):
        print(k, tuple(v.shape), v.dtype)
    else:
        print(k, type(v), v)


In [ ]:
base_model = MultiCamBEVv4SmallRover(
    num_rover_classes=len(rover_vocab),
    rover_emb_dim=cfg['rover_emb_dim'],
    rover_cond_dim=cfg['rover_cond_dim'],
).to(device)

if cfg['use_dp_if_available'] and device.type == 'cuda' and torch.cuda.device_count() > 1:
    print('Wrapping model with DataParallel on', torch.cuda.device_count(), 'GPUs')
    model = nn.DataParallel(base_model)
else:
    model = base_model

core_model = unwrap_model(model)
backbone_params, embed_params, other_params = [], [], []
for name, p in core_model.named_parameters():
    if not p.requires_grad:
        continue
    if name.startswith('backbone.'):
        backbone_params.append(p)
    elif name.startswith('rover_embed.'):
        embed_params.append(p)
    else:
        other_params.append(p)

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': cfg['lr_backbone'], 'weight_decay': cfg['weight_decay']},
    {'params': other_params, 'lr': cfg['lr_head'], 'weight_decay': cfg['weight_decay']},
    {'params': embed_params, 'lr': cfg['lr_head'], 'weight_decay': cfg['embed_weight_decay']},
])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['epochs'], eta_min=1e-6)
loss_fn = CompoundLossV2(pos_weight=cfg['pos_weight']).to(device)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

ema_model = copy.deepcopy(core_model).to(device).eval()
for p in ema_model.parameters():
    p.requires_grad = False

print('trainable params M:', sum(p.numel() for p in core_model.parameters() if p.requires_grad) / 1e6)


In [ ]:
thresholds = [round(x, 2) for x in np.arange(0.30, 0.81, 0.02)]
log = []
best_iou = -1.0
best_ema_iou = -1.0
start = time.time()

for epoch in range(cfg['epochs']):
    model.train()
    loader = loader_warm if epoch < cfg['warmup_epochs'] else loader_train
    phase = 'WARM' if epoch < cfg['warmup_epochs'] else 'AUG'

    losses = []
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(loader, desc=f'ep{epoch:02d} [{phase}]')
    for step, batch in enumerate(pbar):
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)
        gt = batch['gt'].to(device, non_blocking=True)

        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = model(images, intr, c2c, rover_id)
            loss, parts = loss_fn(logits, gt)
            loss = loss / cfg['grad_accum']

        scaler.scale(loss).backward()
        if (step + 1) % cfg['grad_accum'] == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            update_ema(ema_model, model, cfg['ema_decay'])

        losses.append(loss.item() * cfg['grad_accum'])
        if step % 20 == 0:
            pbar.set_postfix(loss=f'{np.mean(losses[-50:]):.3f}', bce=f"{parts['bce']:.2f}")

    scheduler.step()

    val_iou_05 = evaluate_iou(model, loader_val, threshold=0.5)
    val_iou_05_ema = evaluate_iou(ema_model, loader_val, threshold=0.5)
    elapsed = (time.time() - start) / 60

    row = {
        'epoch': epoch,
        'loss': float(np.mean(losses)),
        'val_iou_05': float(val_iou_05),
        'val_iou_05_ema': float(val_iou_05_ema),
        'minutes': float(elapsed),
    }

    if epoch % 2 == 0 or epoch == cfg['epochs'] - 1:
        sweep_model = streaming_threshold_sweep(model, loader_val, thresholds)
        best_t_model, best_iou_model = max(sweep_model.items(), key=lambda kv: kv[1])
        sweep_ema = streaming_threshold_sweep(ema_model, loader_val, thresholds)
        best_t_ema, best_iou_ema = max(sweep_ema.items(), key=lambda kv: kv[1])
        row.update({
            'best_t_model': float(best_t_model),
            'best_iou_model': float(best_iou_model),
            'best_t_ema': float(best_t_ema),
            'best_iou_ema': float(best_iou_ema),
        })
        print(f"ep{epoch:02d} | loss {np.mean(losses):.3f} | val@0.5 {val_iou_05:.4f} | ema@0.5 {val_iou_05_ema:.4f} | best_t {best_t_model:.2f}/{best_t_ema:.2f} | best_iou {best_iou_model:.4f}/{best_iou_ema:.4f} | {elapsed:.1f}m")

        if best_iou_model > best_iou:
            best_iou = best_iou_model
            torch.save({
                'model_type': 'v4_small_rover',
                'model': unwrap_model(model).state_dict(),
                'epoch': epoch,
                'best_iou': best_iou_model,
                'best_t': best_t_model,
                'rover_vocab': rover_vocab,
                'cfg': cfg,
            }, RUN_DIR / 'best.pt')
            print('  new best model:', best_iou_model)

        if best_iou_ema > best_ema_iou:
            best_ema_iou = best_iou_ema
            torch.save({
                'model_type': 'v4_small_rover',
                'model': unwrap_model(model).state_dict(),
                'ema': ema_model.state_dict(),
                'epoch': epoch,
                'best_ema_iou': best_iou_ema,
                'best_t_ema': best_t_ema,
                'rover_vocab': rover_vocab,
                'cfg': cfg,
            }, RUN_DIR / 'ema_best.pt')
            print('  new best ema:', best_iou_ema)
    else:
        print(f"ep{epoch:02d} | loss {np.mean(losses):.3f} | val@0.5 {val_iou_05:.4f} | ema@0.5 {val_iou_05_ema:.4f} | {elapsed:.1f}m")

    log.append(row)
    pd.DataFrame(log).to_csv(RUN_DIR / 'log.csv', index=False)
    torch.save({
        'model_type': 'v4_small_rover',
        'model': unwrap_model(model).state_dict(),
        'ema': ema_model.state_dict(),
        'epoch': epoch,
        'rover_vocab': rover_vocab,
        'cfg': cfg,
    }, RUN_DIR / 'last.pt')


In [ ]:
ckpt = torch.load(RUN_DIR / 'ema_best.pt', map_location=device) if (RUN_DIR / 'ema_best.pt').exists() else torch.load(RUN_DIR / 'best.pt', map_location=device)
model_eval = MultiCamBEVv4SmallRover(
    num_rover_classes=len(rover_vocab),
    rover_emb_dim=cfg['rover_emb_dim'],
    rover_cond_dim=cfg['rover_cond_dim'],
).to(device)
state = ckpt['ema'] if 'ema' in ckpt else ckpt['model']
model_eval.load_state_dict(state, strict=False)

thresholds = [round(x, 2) for x in np.arange(0.20, 0.81, 0.02)]
iou_by_t = streaming_threshold_sweep(model_eval, loader_val, thresholds)
best_t, best_iou = max(iou_by_t.items(), key=lambda kv: kv[1])
print('best threshold:', best_t, 'best_iou:', best_iou)
for t, iou in iou_by_t.items():
    marker = ' <- best' if t == best_t else ''
    print(f't={t:.2f}  iou={iou:.4f}{marker}')


## Notes

- This notebook is the direct no-cleaning ablation against the cleaned pipeline.
- The split still uses `train + val` merged into one pool and holds out about 200 validation samples in a test-matched group-aware way.
- Rare and unseen rovers map to `Other=0`.
- When 2 CUDA GPUs are visible, training uses `nn.DataParallel`; checkpoints are always saved from the unwrapped base model.
